# ROI Lens: Marketing Attribution & Budget Optimization
### Nexus Consumer Brands â€” Q1 2026 Analysis

**Objective:** Replace legacy last-click attribution with a probabilistic multi-touch framework (Markov Chain + Shapley Value) and optimize the Rs.100 Crore quarterly budget across 10 brands and 5 channels.

---

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.dpi'] = 120

RESULTS = '../outputs/results/'
FIGURES = '../outputs/figures/'
print('Setup complete.')

---
## Phase 1: Data Loading & Bot Detection

We loaded three datasets:
- **touchpoints.csv** (566K rows) â€” every user interaction across channels
- **user_profiles.csv** (100K rows) â€” persona segments mapped to Ai Palette trends
- **campaign_spend.csv** (50 rows) â€” budget and pricing per brand Ã— channel

### 1.1 Data Profile

In [ ]:
# Load clean data from Phase 1
df_clean = pd.read_csv(f'{RESULTS}touchpoints_clean.csv', parse_dates=['Timestamp'])
df_spend = pd.read_csv('../data/campaign_spend.csv')
df_profiles = pd.read_csv('../data/user_profiles.csv')

print(f'Clean dataset: {len(df_clean):,} rows x {len(df_clean.columns)} columns')
print(f'Unique users : {df_clean["User_ID"].nunique():,}')
print(f'Brands       : {sorted(df_clean["Brand_ID"].unique())}')
print(f'Channels     : {sorted(df_clean["Channel"].unique())}')
print(f'Date range   : {df_clean["Timestamp"].min()} to {df_clean["Timestamp"].max()}')
print(f'\nEvent distribution:')
display(df_clean['Event_Type'].value_counts().to_frame('Count'))

### 1.2 Bot Detection Results

We detected **1,371 bot users** (1.37% of all users) who generated **131,874 touchpoints** â€” a staggering **24% of all traffic**. These bots were identified by their 24/7 activity pattern (active across 20+ hours daily). Bot traffic was uniformly distributed across all channels (~24% each), meaning no single channel was disproportionately polluted.

In [ ]:
display(Image(filename=f'{FIGURES}01_bot_detection.png', width=800))

**Key Insight:** Without bot removal, attribution models would assign inflated credit to channels based on fake impressions. Cleaning this 24% of noise was essential before any analysis.

---
## Phase 2: Funnel Analysis & Last-Click Baseline

### 2.1 Conversion Funnel

The overall funnel shows steep drop-offs at each stage:

In [ ]:
display(Image(filename=f'{FIGURES}02_conversion_funnel.png', width=800))

### 2.2 Brand-Level Performance

Not all brands convert equally â€” B07 and B02 significantly outperform the rest:

In [ ]:
display(Image(filename=f'{FIGURES}03_brand_conversions.png', width=800))

In [ ]:
# Brand funnel data
funnel = pd.read_csv(f'{RESULTS}funnel_brand.csv')
funnel = funnel.sort_values('Purchases', ascending=False)
display(funnel[['Brand_ID', 'Impressions', 'Clicks', 'CTR(%)', 'Purchases', 'Conv_Rate(%)']])

### 2.3 Last-Click Attribution (The Flawed Baseline)

Under last-click, 100% of credit goes to the last channel clicked before purchase. A critical finding: **78.3% of purchases were "orphans"** â€” users who purchased without any recorded click. This alone proves last-click is fundamentally broken for this dataset.

In [ ]:
lc_agg = pd.read_csv(f'{RESULTS}last_click_aggregated.csv')
lc_fin = pd.read_csv(f'{RESULTS}last_click_financials.csv')

# Show top distortions
top_distorted = lc_agg.sort_values('LC_Share(%)', ascending=False).head(10)
print('Top 10 Last-Click credit allocations (most concentrated):')
display(top_distorted[['Brand_ID', 'Channel', 'LC_Conversions', 'LC_Share(%)']])

---
## Phase 3: Multi-Touch Attribution

We built two complementary probabilistic models to find the **true** channel contributions:

1. **Markov Chain** â€” Models user journeys as a stochastic process. Computes each channel's "removal effect" (how much conversion drops if that channel is removed from all paths).

2. **Shapley Value** â€” Game-theoretic approach that computes each channel's fair contribution across all possible coalitions (2^5 = 32 subsets per brand).

### 3.1 Attribution Comparison

Here we compare all three models for the most interesting brands:

In [ ]:
display(Image(filename=f'{FIGURES}04_attribution_comparison.png', width=900))

**What this tells us:**
- **B01:** Last-click gives Instagram 58% credit, but Markov says it's only 33%. Instagram is the "closer" but other channels are doing the awareness work.
- **B02:** Google Search gets 72% under last-click but only 37% under Markov. Influencer Blog and Marketplace are being massively under-credited.
- **B07:** Same pattern â€” Google Search is over-credited; YouTube and Influencer Blog deserve far more.

### 3.2 Attribution Shift Heatmap

This heatmap shows exactly where last-click is wrong. **Red = over-credited**, **Green = under-credited** by last-click:

In [ ]:
display(Image(filename=f'{FIGURES}05_attribution_heatmap.png', width=800))

**Critical findings:**
- B02 Google Search is over-credited by **-35.4 percentage points**
- B09 Marketplace is over-credited by **-31.8 points**
- B05 YouTube is under-credited by **+12.7 points** â€” it's a hidden performer

### 3.3 Channel Funnel Roles

Based on positional analysis (where channels appear in converting journeys), we classified each channel's funnel role:

In [ ]:
display(Image(filename=f'{FIGURES}06_channel_roles.png', width=800))

In [ ]:
roles = pd.read_csv(f'{RESULTS}channel_roles.csv')
# Show closers and primers
print('Closers (drive final purchase):')
display(roles[roles['Funnel_Role'] == 'Closer/Converter'][['Brand_ID', 'Channel', 'Last_Touch(%)', 'Markov_Attribution(%)']])
print('\nPrimers (introduce users to funnel):')
display(roles[roles['Funnel_Role'] == 'Primer/Introducer'][['Brand_ID', 'Channel', 'First_Touch(%)', 'Markov_Attribution(%)']])

---
## Phase 4: Financial Layer â€” True CPA

### 4.1 CPA Comparison: Last-Click vs Markov

When we recalculate Cost Per Acquisition using Markov-attributed conversions instead of last-click, the picture changes dramatically:

In [ ]:
display(Image(filename=f'{FIGURES}07_cpa_comparison.png', width=800))

In [ ]:
cpa = pd.read_csv(f'{RESULTS}cpa_comparison.csv')
# Show biggest CPA shifts
cpa_sorted = cpa.reindex(cpa['CPA_Shift_Markov(%)'].abs().sort_values(ascending=False).index).head(10)
print('Top 10 biggest CPA shifts (Last-Click vs Markov):')
display(cpa_sorted[['Brand_ID', 'Channel', 'CPA_LastClick', 'CPA_Markov', 'CPA_Shift_Markov(%)']])

### 4.2 Ad Fatigue & Diminishing Returns

We fitted log-response curves to each brand Ã— channel to quantify diminishing returns. The "fatigue ratio" measures how much CPA increases from 25% to 100% of budget spend:

In [ ]:
curves = pd.read_csv(f'{RESULTS}response_curves.csv')

# Show response curve for one brand
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
brands_to_show = ['B01', 'B02', 'B05', 'B07', 'B08', 'B09']
colors = {'Google Search': '#4285F4', 'Instagram': '#E1306C', 'YouTube': '#FF0000',
          'Influencer Blog': '#06B6D4', 'Marketplace': '#F59E0B'}

for ax, brand in zip(axes.flat, brands_to_show):
    brand_data = curves[curves['Brand_ID'] == brand]
    for ch in brand_data['Channel'].unique():
        ch_data = brand_data[brand_data['Channel'] == ch]
        ax.plot(ch_data['Cumulative_Spend']/1e7, ch_data['Cumulative_Conversions'],
                'o-', label=ch.replace('Influencer Blog', 'Influencer'), 
                color=colors.get(ch, '#666'), markersize=3, linewidth=1.5)
    ax.set_title(brand, fontweight='bold')
    ax.set_xlabel('Spend (Rs. Cr)')
    ax.set_ylabel('Cumulative Conv')
    ax.grid(True, alpha=0.3)

axes[0][0].legend(fontsize=7, loc='upper left')
fig.suptitle('Response Curves: Spend vs Conversions (Diminishing Returns Visible)', 
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

The curves flatten as spend increases â€” classic diminishing returns. This non-linearity is why we use saturation models, not linear projections, for budget optimization.

---
## Phase 5: Budget Optimization

### 5.1 The Rs.100 Crore Reallocation

Using scipy's SLSQP optimizer with the fitted response curves, we solved for the budget split that maximizes total conversions under constraints:
- Budget per brand = Rs.10 Crore
- Each channel gets between 5% and 50% of brand budget

### Budget Shift Recommendations:

In [ ]:
display(Image(filename=f'{FIGURES}08_budget_reallocation.png', width=800))

### 5.2 Expected Conversion Lift per Brand

In [ ]:
display(Image(filename=f'{FIGURES}09_conversion_lift.png', width=800))

In [ ]:
opt = pd.read_csv(f'{RESULTS}budget_optimization.csv')

# Overall impact
total_current = opt['Current_Conv'].sum()
total_expected = opt['Expected_Conv'].sum()
print(f'Current total conversions : {total_current:,.0f}')
print(f'Optimized conversions     : {total_expected:,.0f}')
print(f'Conversion lift           : +{(total_expected-total_current)/total_current*100:.1f}%')
print(f'Budget                    : Rs.100 Crore (unchanged)')
print(f'\n=> {total_expected - total_current:.0f} additional conversions at ZERO incremental cost')

In [ ]:
# Detailed reallocation table
opt_display = opt.copy()
opt_display['Current_Spend'] = opt_display['Current_Spend'].apply(lambda x: f'Rs.{x/1e7:.2f}Cr')
opt_display['Optimized_Spend'] = opt_display['Optimized_Spend'].apply(lambda x: f'Rs.{x/1e7:.2f}Cr')
opt_display['Delta_Spend'] = opt_display['Delta_Spend(%)'].apply(lambda x: f'{x:+.0f}%')

print('Full Reallocation Strategy:')
display(opt_display[['Brand_ID', 'Channel', 'Current_Spend', 'Optimized_Spend', 'Delta_Spend',
                      'Current_Conv', 'Expected_Conv']])

### 5.3 Sensitivity Analysis

In [ ]:
display(Image(filename=f'{FIGURES}10_sensitivity_analysis.png', width=800))

In [ ]:
sens = pd.read_csv(f'{RESULTS}sensitivity_analysis.csv')
display(sens)

Marginal CPA rises from Rs.5.95L at Rs.80Cr to Rs.11.1L at Rs.140Cr â€” clear diminishing returns at portfolio level.

---
## Executive Summary & Recommendations

### Finding 1: Last-click attribution is costing Nexus money
- Last-click over-credits "closer" channels (Google Search, Marketplace) by up to **35 percentage points**
- Upper-funnel channels (YouTube, Influencer Blog) are systematically under-valued
- 78% of purchases had no prior click â€” last-click literally cannot attribute most conversions

### Finding 2: Significant budget is wasted on saturated channels
- Every channel shows ~3x ad fatigue (CPA triples from 25% to 100% of budget)
- B05 YouTube: Rs.40Cr spent for just 4 conversions (Rs.1Cr CPA per conversion)
- B03 YouTube: Rs.49Cr spent â€” the most over-invested channel-brand pair

### Finding 3: Reallocation unlocks +17.4% more conversions for free
- **955 additional conversions** from the same Rs.100Cr budget
- B01 stands to gain the most (+63% lift) by properly funding Instagram
- B09 is the only brand that may see slight decline (-2.6%) due to Marketplace saturation

### Recommended Actions
1. **Defund:** B03 YouTube (-Rs.29Cr), B07 Instagram (-Rs.28Cr), B09 Marketplace (-Rs.27Cr)
2. **Invest:** B01 Instagram (+Rs.19Cr), B06 Instagram (+Rs.19Cr), B02 Influencer Blog (+Rs.18Cr)
3. **Frequency Cap:** Channels hitting the 50% budget ceiling need creative refresh, not more spend
4. **Next Steps:** A/B test the Markov-based allocation on 2 brands before full rollout

---
*ROI Lens â€” Built for Nexus Consumer Brands | Data-driven attribution intelligence*